In [1]:
# 1. Construct the Neutrino client. Read only the PAT from ./config.json.
import json
from pathlib import Path

from dss_client import NeutrinoClient

PAT = json.loads(Path("../config.json").read_text(encoding="utf-8"))["pat"]
client = NeutrinoClient.from_pat(
    host="dsa-test.qa6.us-west-2.aws.snowflakecomputing.com",
    pat=PAT,
    database="NEUTRINO_DB",
    schema="PUBLIC",
    endpoint="cortex-training",
    poll_interval=0.5,
    poll_timeout=1800.0,
    verify_ssl=True,
)
del PAT

In [2]:
# 2. Construct a single-GPU Qwen3-8B sampling job.
job_body = {
    "sub_job_configs": [
        {
            "job_type": "sampling",
            "model_name": "Qwen/Qwen3-8B",
            "dtype": "bfloat16",
            "seed": 42,
            "inference_config": {
                "max_seq_len": 2048,
                "n_gpus": 1,
                "gpu_memory_utilization": 0.8,
            },
        }
    ]
}

In [3]:
# 3. Submit the job and wait for the sampling worker to start (will take about 3 min).
job_id = client.create_job_from_body(job_body)["job_id"]
print(f"Job {job_id} is starting")
client.wait_for_job(job_id)
print(f"Job {job_id} is running")

Job 10bf4d7a-e9d8-4c3f-b6e1-bb2cacb7b6fb is starting
Job 10bf4d7a-e9d8-4c3f-b6e1-bb2cacb7b6fb is running


In [4]:
# 4. Generate text and wait for the asynchronous request to finish.
request_id = client.generate(
    job_id,
    prompts=["Explain in one paragraph why snowflakes have six-fold symmetry."],
    sampling_params={"max_tokens": 128, "temperature": 0.7},
)
result = client.poll_request(job_id, request_id)
print(result['results'][0]['text'])

 Snowflakes exhibit six-fold symmetry due to the molecular structure of water and the way it crystallizes. When water freezes, its molecules arrange themselves into a hexagonal lattice, a structure that is energetically favorable because it allows for optimal hydrogen bonding between water molecules. This hexagonal arrangement leads to the formation of a six-sided (hexagonal) crystal structure, which is why snowflakes typically display six-fold symmetry. As the snowflake grows, each of the six arms develops in a similar manner, influenced by temperature and humidity conditions, resulting in the intricate and symmetrical patterns observed in snowflakes. The underlying hexagonal symmetry


In [5]:
# 5. Tear down the sampling job and release its GPU.
client.cancel_job(job_id)
print(f"Cancelled job {job_id}")
job_id = None

Cancelled job 10bf4d7a-e9d8-4c3f-b6e1-bb2cacb7b6fb
